# Hybrid and step systems

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/06_hybrid.ipynb)


> **Architecture note:** Continuous `DynamicSystem` block diagrams are minilink's core abstraction. The discrete-in-the-loop stack (`StepSystem`, `Computer`, `HybridDiagram`, `HybridSimulator`) provides simulation orchestration for digital controllers and quick MPC demos. It is a provisional utility rather than a core diagram `System` (promotion to an official `HybridLoop` is planned for v1.0).

Official API intro to the discrete-in-the-loop stack:

1. **`StepSystem`** — a discrete leaf, $x_{k+1} = \mathrm{step}(x_k, u_k)$
2. **`StepDiagramSystem`** — step leaves wired together: a digital loop
3. **`Computer`** — a step side scheduled at a period: `block % dt`
4. **`HybridDiagram`** — `computer @ plant`: the computer ticks, the continuous plant integrates between ticks (sample, then zero-order hold)

**Scripts for depth:** `examples/demos/hybrid/` (multi-rate schedules, sampled sliding mode) · `examples/demos/mpc/` (an MPC is a `Computer`) · `examples/projects/mpc/` (spatial MPC stack).

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. A discrete leaf

A `StepSystem` defines the recurrence in `step`; with `output_dim == n` the output is the state. `compute_rollout` iterates it for a constant input and `plot_rollout` draws the samples.

In [ ]:
import numpy as np

from minilink import StepSystem


class Accumulator(StepSystem):
    def __init__(self):
        super().__init__(n=1, input_dim=1, output_dim=1, y_dependencies=())

    def step(self, x, u, k=0, params=None):
        return np.array([x[0] + u[0]])


acc = Accumulator()
acc.compute_rollout(n_steps=10, u=np.array([1.0]))
acc.plot_rollout()

## 2. A digital loop

Step leaves compose like continuous ones. A proportional controller and the accumulator in unity feedback: every tick the controller reads $y_k$, the plant advances with $u_k$.

In [ ]:
from minilink import ProportionalController, StepDiagramSystem

loop = StepDiagramSystem()
loop.add_subsystem(ProportionalController(0.3), "ctl")
loop.add_subsystem(Accumulator(), "plant")
loop.add_input_port("r")
loop.connect("input", "r", "ctl", "r")
loop.connect("plant", "y", "ctl", "y")
loop.connect("ctl", "u", "plant", "u")
loop.connect_new_output_port("plant", "y", "y")

loop.plot_diagram()
loop.compute_rollout(n_steps=40, u=np.array([1.0]))
loop.plot_rollout()

## 3. A digital controller on a continuous plant

`block % dt` turns any controller into a `Computer` that ticks every `dt` seconds; `computer @ plant` samples the plant output at each tick and holds the command until the next one, while the plant integrates in continuous time in between. The same operator wires an MPC (`mpc @ plant`).

In [ ]:
from minilink import Integrator

computer = ProportionalController(2.0) % 0.1  # ticks every 0.1 s
hybrid = computer @ Integrator()  # sample y, hold u between ticks
hybrid.plot_diagram()

result = hybrid.compute_forced(lambda t: np.array([1.0]), input_port_id="r", tf=4.0)
hybrid.plot_trajectory(signals=("r", "y", "u"))

## Where next

- Two rates on one computer (`StepSchedule.from_rates`): `examples/demos/hybrid/hybrid_multi_rate.py`
- A sampled sliding-mode controller: `examples/demos/hybrid/sampled_smc_pendulum.py`
- MPC as a computer, NumPy only: `examples/demos/mpc/mpc_integrator_numpy.py`; JAX and a car: `mpc_car_minimal.py`
- The full spatial MPC stack (research lane): `examples/projects/mpc/mpc_spatial_stack.ipynb`